# Tratamento da Tabela

In [ ]:
#!pip install pyspark

## Importação de Bibliotecas

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import pandas as pd
from pyspark.sql.functions import split, col, lit

spark = SparkSession.builder.master("local[*]").getOrCreate()

## Carregar Tabela

In [ ]:
df2022_pd = pd.read_excel("/content/BASE DE DADOS PEDE 2024 - DATATHON.xlsx", sheet_name="PEDE2022")
df2023_pd = pd.read_excel("/content/BASE DE DADOS PEDE 2024 - DATATHON.xlsx", sheet_name="PEDE2023")
df2024_pd = pd.read_excel("/content/BASE DE DADOS PEDE 2024 - DATATHON.xlsx", sheet_name="PEDE2024")

In [ ]:
df2022 = spark.createDataFrame(df2022_pd.astype(str))
df2023 = spark.createDataFrame(df2023_pd.astype(str))
df2024 = spark.createDataFrame(df2024_pd.astype(str))

## Análise Exploratória dos Dados

In [ ]:
df2022.show(10, False)
df2023.show(10, False)
df2024.show(10, False)

+-----+----+-----+--------+--------+--------+------+------------+---------------------+--------+--------+--------+-------+---+---+---+-----+-----------+---------------------+------------+-------------------------+------------+-------------------------+------------+-------------------------+---+---+---+----------------+---+-----+------+------+--------+----------+-----+----+-----------------------+-----+---------------------------------------------------+--------------------------------------------------+-------------------------------------------------------------+
|RA   |Fase|Turma|Nome    |Ano nasc|Idade 22|Gênero|Ano ingresso|Instituição de ensino|Pedra 20|Pedra 21|Pedra 22|INDE 22|Cg |Cf |Ct |Nº Av|Avaliador1 |Rec Av1              |Avaliador2  |Rec Av2                  |Avaliador3  |Rec Av3                  |Avaliador4  |Rec Av4                  |IAA|IEG|IPS|Rec Psicologia  |IDA|Matem|Portug|Inglês|Indicado|Atingiu PV|IPV  |IAN |Fase ideal             |Defas|Destaque IEG            

In [ ]:
print(f"Colunas no df2022: {len(df2022.columns)}")
print(f"Linhas no df2022: {df2022.count()}\n")

print(f"Colunas no df2023: {len(df2023.columns)}")
print(f"Linhas no df2023: {df2023.count()}\n")

print(f"Colunas no df2024: {len(df2024.columns)}")
print(f"Linhas no df2024: {df2024.count()}")

Colunas no df2022: 42
Linhas no df2022: 860

Colunas no df2023: 48
Linhas no df2023: 1014

Colunas no df2024: 50
Linhas no df2024: 1156


In [ ]:
df2022.printSchema()
df2023.printSchema()
df2024.printSchema()

root
 |-- RA: string (nullable = true)
 |-- Fase: string (nullable = true)
 |-- Turma: string (nullable = true)
 |-- Nome: string (nullable = true)
 |-- Ano nasc: string (nullable = true)
 |-- Idade 22: string (nullable = true)
 |-- Gênero: string (nullable = true)
 |-- Ano ingresso: string (nullable = true)
 |-- Instituição de ensino: string (nullable = true)
 |-- Pedra 20: string (nullable = true)
 |-- Pedra 21: string (nullable = true)
 |-- Pedra 22: string (nullable = true)
 |-- INDE 22: string (nullable = true)
 |-- Cg: string (nullable = true)
 |-- Cf: string (nullable = true)
 |-- Ct: string (nullable = true)
 |-- Nº Av: string (nullable = true)
 |-- Avaliador1: string (nullable = true)
 |-- Rec Av1: string (nullable = true)
 |-- Avaliador2: string (nullable = true)
 |-- Rec Av2: string (nullable = true)
 |-- Avaliador3: string (nullable = true)
 |-- Rec Av3: string (nullable = true)
 |-- Avaliador4: string (nullable = true)
 |-- Rec Av4: string (nullable = true)
 |-- IAA: strin

### Validação de Colunas

In [ ]:
colunas_escolhidas = [
    "IAA", "IDA", "IEG", "IPP", "IAN", "IPS", "IPV", "RA",
    "Pedra 20", "Pedra 21", "Pedra 22", "Pedra 2023", "Pedra 2024",
    "INDE 22", "Rec Psicologia", "Atingiu PV", "Defas",
    "Destaque IEG", "Destaque IDA", "Destaque IPV"
]

dfs = {
    "2022": df2022,
    "2023": df2023,
    "2024": df2024
}

for col in colunas_escolhidas:
    existe_em = [ano for ano, df in dfs.items() if col in df.columns]
    print(f"{col}: {existe_em}")


IAA: ['2022', '2023', '2024']
IDA: ['2022', '2023', '2024']
IEG: ['2022', '2023', '2024']
IPP: ['2023', '2024']
IAN: ['2022', '2023', '2024']
IPS: ['2022', '2023', '2024']
IPV: ['2022', '2023', '2024']
RA: ['2022', '2023', '2024']
Pedra 20: ['2022', '2023', '2024']
Pedra 21: ['2022', '2023', '2024']
Pedra 22: ['2022', '2023', '2024']
Pedra 2023: ['2023']
Pedra 2024: ['2024']
INDE 22: ['2022', '2023', '2024']
Rec Psicologia: ['2022', '2023', '2024']
Atingiu PV: ['2022', '2023', '2024']
Defas: ['2022']
Destaque IEG: ['2022', '2023', '2024']
Destaque IDA: ['2022', '2023', '2024']
Destaque IPV: ['2022', '2023', '2024']


### Colunas Exclusivas de cada DF

In [ ]:
exclusivas_2022 = set(df2022.columns) - set(df2023.columns) - set(df2024.columns)
exclusivas_2023 = set(df2023.columns) - set(df2022.columns) - set(df2024.columns)
exclusivas_2024 = set(df2024.columns) - set(df2022.columns) - set(df2023.columns)

print("Colunas Únicas em 2022:", exclusivas_2022)
print("Colunas Únicas em 2023:", exclusivas_2023)
print("Colunas Únicas em 2024:", exclusivas_2024)

Colunas Únicas em 2022: {'Defas', 'Portug', 'Inglês', 'Ano nasc', 'Idade 22', 'Matem', 'Fase ideal', 'Nome'}
Colunas Únicas em 2023: {'Destaque IPV.1', 'INDE 2023', 'Pedra 2023'}
Colunas Únicas em 2024: {'Ativo/ Inativo', 'Avaliador5', 'Avaliador6', 'INDE 2024', 'Ativo/ Inativo.1', 'Pedra 2024', 'Escola'}


## Tratamento dos Dados

### Renomear Colunas

In [ ]:
renome_colunas = {
    "IAA": "iaa_autoavaliacao",
    "IDA": "ida_desempenho_academico",
    "IEG": "ieg_engajamento_atividades",
    "IPP": "ipp_psicopedagogicos",
    "IAN": "ian_adequacao_nivel",
    "IPS": "ips_aspectos_psicossociais",
    "IPV": "ipv_ponto_virada",
    "RA": "ra",
    "Pedra 20": "pedra_2020",
    "Pedra 21": "pedra_2021",
    "Pedra 22": "pedra_2022",
    "Pedra 2023": "pedra_2023",
    "Pedra 2024": "pedra_2024",
    "INDE 22": "inde_indice_desen_educacional",
    "Rec Psicologia": "requer_psicologia",
    "Atingiu PV": "atingiu_pv",
    "Defas": "defasagem",
    "Destaque IEG": "destaque_ieg",
    "Destaque IDA": "destaque_ida",
    "Destaque IPV": "destaque_ipv"
}


In [ ]:
def padronizar_df_spark(df, colunas_escolhidas, renome_colunas, ano):

    # Garante todas as colunas
    select_cols = [
        col(f"`{c}`") if c in df.columns else lit(None).alias(c)
        for c in colunas_escolhidas
    ]

    df = df.select(select_cols)

    # Renomeia colunas
    df = df.select([
        col(c).alias(renome_colunas.get(c, c))
        for c in df.columns
    ])

    # Adiciona coluna de Ano para identificar o arquivo depois do append
    df = df.withColumn("ano_arquivo", lit(ano))

    return df


In [ ]:
from pyspark.sql.functions import col, lit

df2022_colunas = padronizar_df_spark(df2022, colunas_escolhidas, renome_colunas, 2022)
df2023_colunas = padronizar_df_spark(df2023, colunas_escolhidas, renome_colunas, 2023)
df2024_colunas = padronizar_df_spark(df2024, colunas_escolhidas, renome_colunas, 2024)

In [ ]:
df_final = df2022_colunas.unionByName(df2023_colunas).unionByName(df2024_colunas)

In [ ]:
print(f"Número de colunas: {len(df_final.columns)}")
print(f"Número de linhas: {df_final.count()}\n")

df_final.show(10, False)

Número de colunas: 21
Número de linhas: 3030

+-----------------+------------------------+--------------------------+--------------------+-------------------+--------------------------+----------------+-----+----------+----------+----------+----------+----------+-----------------------------+-----------------+----------+---------+---------------------------------------------------+--------------------------------------------------+-------------------------------------------------------------+-----------+
|iaa_autoavaliacao|ida_desempenho_academico|ieg_engajamento_atividades|ipp_psicopedagogicos|ian_adequacao_nivel|ips_aspectos_psicossociais|ipv_ponto_virada|ra   |pedra_2020|pedra_2021|pedra_2022|pedra_2023|pedra_2024|inde_indice_desen_educacional|requer_psicologia|atingiu_pv|defasagem|destaque_ieg                                       |destaque_ida                                      |destaque_ipv                                                 |ano_arquivo|
+-----------------+-------

In [ ]:
df_final = df_final.select([
    F.when(
        F.lower(F.trim(F.col(c).cast("string"))).isin("nan", "none", ""),
        None
    ).otherwise(F.col(c)).alias(c)
    for c in df_final.columns
])

### Reordenar Colunas

In [ ]:
ordem_final = [
    "ra",
    "iaa_autoavaliacao",
    "ida_desempenho_academico",
    "ieg_engajamento_atividades",
    "ipp_psicopedagogicos",
    "ian_adequacao_nivel",
    "ips_aspectos_psicossociais",
    "ipv_ponto_virada",
    "pedra_2020",
    "pedra_2021",
    "pedra_2022",
    "pedra_2023",
    "pedra_2024",
    "inde_indice_desen_educacional",
    "requer_psicologia",
    "atingiu_pv",
    "defasagem",
    "destaque_ieg",
    "destaque_ida",
    "destaque_ipv",
    "ano_arquivo"
]

In [ ]:
df_final = df_final.select(ordem_final)

df_final.show(10, False)

+-----+-----------------+------------------------+--------------------------+--------------------+-------------------+--------------------------+----------------+----------+----------+----------+----------+----------+-----------------------------+-----------------+----------+---------+---------------------------------------------------+--------------------------------------------------+-------------------------------------------------------------+-----------+
|ra   |iaa_autoavaliacao|ida_desempenho_academico|ieg_engajamento_atividades|ipp_psicopedagogicos|ian_adequacao_nivel|ips_aspectos_psicossociais|ipv_ponto_virada|pedra_2020|pedra_2021|pedra_2022|pedra_2023|pedra_2024|inde_indice_desen_educacional|requer_psicologia|atingiu_pv|defasagem|destaque_ieg                                       |destaque_ida                                      |destaque_ipv                                                 |ano_arquivo|
+-----+-----------------+------------------------+----------------------

### Normalização das Colunas

In [ ]:
colunas = ["destaque_ieg", "destaque_ida", "destaque_ipv"]

for c in colunas:
    df_final = df_final.withColumn(c, split(col(c), ":").getItem(0))


df_final.show(10, False)

+-----+-----------------+------------------------+--------------------------+--------------------+-------------------+--------------------------+----------------+----------+----------+----------+----------+----------+-----------------------------+-----------------+----------+---------+------------+------------+------------+-----------+
|ra   |iaa_autoavaliacao|ida_desempenho_academico|ieg_engajamento_atividades|ipp_psicopedagogicos|ian_adequacao_nivel|ips_aspectos_psicossociais|ipv_ponto_virada|pedra_2020|pedra_2021|pedra_2022|pedra_2023|pedra_2024|inde_indice_desen_educacional|requer_psicologia|atingiu_pv|defasagem|destaque_ieg|destaque_ida|destaque_ipv|ano_arquivo|
+-----+-----------------+------------------------+--------------------------+--------------------+-------------------+--------------------------+----------------+----------+----------+----------+----------+----------+-----------------------------+-----------------+----------+---------+------------+------------+------------

Validação

In [ ]:
def perfil_nulos(df):

    total_linhas = df.count()

    resultado = df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}_nulls")
        for c in df.columns
    ])

    resultado_percent = df.select([
        F.round(
            F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) / total_linhas * 100,
            2
        ).alias(f"{c}_pct_nulls")
        for c in df.columns
    ])

    resultado.show(truncate=False)
    resultado_percent.show(truncate=False)

perfil_nulos(df_final)

+--------+-----------------------+------------------------------+--------------------------------+--------------------------+-------------------------+--------------------------------+----------------------+----------------+----------------+----------------+----------------+----------------+-----------------------------------+-----------------------+----------------+---------------+------------------+------------------+------------------+-----------------+
|ra_nulls|iaa_autoavaliacao_nulls|ida_desempenho_academico_nulls|ieg_engajamento_atividades_nulls|ipp_psicopedagogicos_nulls|ian_adequacao_nivel_nulls|ips_aspectos_psicossociais_nulls|ipv_ponto_virada_nulls|pedra_2020_nulls|pedra_2021_nulls|pedra_2022_nulls|pedra_2023_nulls|pedra_2024_nulls|inde_indice_desen_educacional_nulls|requer_psicologia_nulls|atingiu_pv_nulls|defasagem_nulls|destaque_ieg_nulls|destaque_ida_nulls|destaque_ipv_nulls|ano_arquivo_nulls|
+--------+-----------------------+------------------------------+-------------

### Casting das Colunas

In [ ]:
colunas_numericas = [
    "iaa_autoavaliacao",
    "ida_desempenho_academico",
    "ieg_engajamento_atividades",
    "ipp_psicopedagogicos",
    "ian_adequacao_nivel",
    "ips_aspectos_psicossociais",
    "ipv_ponto_virada",
    "inde_indice_desen_educacional",
    "defasagem"
]

df_final = df_final.select([
    F.round(F.col(c).cast("double"), 2).alias(c) if c in colunas_numericas
    else F.col(c).cast("string").alias(c)
    for c in df_final.columns
])

In [ ]:
df_final.printSchema()

root
 |-- ra: string (nullable = true)
 |-- iaa_autoavaliacao: double (nullable = true)
 |-- ida_desempenho_academico: double (nullable = true)
 |-- ieg_engajamento_atividades: double (nullable = true)
 |-- ipp_psicopedagogicos: double (nullable = true)
 |-- ian_adequacao_nivel: double (nullable = true)
 |-- ips_aspectos_psicossociais: double (nullable = true)
 |-- ipv_ponto_virada: double (nullable = true)
 |-- pedra_2020: string (nullable = true)
 |-- pedra_2021: string (nullable = true)
 |-- pedra_2022: string (nullable = true)
 |-- pedra_2023: string (nullable = true)
 |-- pedra_2024: string (nullable = true)
 |-- inde_indice_desen_educacional: double (nullable = true)
 |-- requer_psicologia: string (nullable = true)
 |-- atingiu_pv: string (nullable = true)
 |-- defasagem: double (nullable = true)
 |-- destaque_ieg: string (nullable = true)
 |-- destaque_ida: string (nullable = true)
 |-- destaque_ipv: string (nullable = true)
 |-- ano_arquivo: string (nullable = true)



In [ ]:
df_final.show(10, False)

+-----+-----------------+------------------------+--------------------------+--------------------+-------------------+--------------------------+----------------+----------+----------+----------+----------+----------+-----------------------------+-----------------+----------+---------+------------+------------+------------+-----------+
|ra   |iaa_autoavaliacao|ida_desempenho_academico|ieg_engajamento_atividades|ipp_psicopedagogicos|ian_adequacao_nivel|ips_aspectos_psicossociais|ipv_ponto_virada|pedra_2020|pedra_2021|pedra_2022|pedra_2023|pedra_2024|inde_indice_desen_educacional|requer_psicologia|atingiu_pv|defasagem|destaque_ieg|destaque_ida|destaque_ipv|ano_arquivo|
+-----+-----------------+------------------------+--------------------------+--------------------+-------------------+--------------------------+----------------+----------+----------+----------+----------+----------+-----------------------------+-----------------+----------+---------+------------+------------+------------

## Exportação Dataframe Final

In [ ]:
df_pandas = df_final.toPandas()

df_pandas.to_excel("/content/fat_datathon.xlsx", index=False)